# Event Replay with TraceLens

When optimizing GPU performance in deep learning, it's often necessary to isolate and benchmark individual operations to pinpoint bottlenecks. Doing this directly from complex model code or large profile traces can be tedious and error-prone.

**Event Replay** in TraceLens solves this by extracting PyTorch operations from a profile trace and replaying them in isolation using a minimal, portable Intermediate Representation (IR). This lets you reproduce, analyze, and benchmark specific operators without needing access to the original model.


In [5]:
import time
import os
import json
import torch
from torch.profiler import profile, record_function, ProfilerActivity

from TraceLens import TreePerfAnalyzer, EventReplayer

In [ ]:
# create a working directory for replay artifacts
def create_dir_and_cwd(dir_name):
    if not os.path.exists(dir_name):
        os.makedirs(dir_name)
    os.chdir(dir_name)
    print(f"Changed working directory to: {os.getcwd()}")

create_dir_and_cwd("event_replay_wd")

## Load the Profile

We already have a trace collected from a ResNet training run. Let's load it with `TreePerfAnalyzer`.

In [ ]:
profile_path = "HPCTrainingExamples/MLExamples/PyTorch_Profiling/trace_0_20_0.json"
perf_analyzer = TreePerfAnalyzer.from_file(profile_path)

Building tree with add_python_func=False
Building CPU op tree with add_python_func=False


## Identify Events of Interest

Let's look at the unique kernel-launching ops to pick an event to replay. We can use the `get_df_kernel_launchers_unique_args` to see which ops consume the most GPU time.

In [8]:
df_kernel_launchers = perf_analyzer.get_df_kernel_launchers(include_kernel_details=True)

# get unique argument combinations with percentage of total GPU time
df_kernel_launchers_unique_args = perf_analyzer.get_df_kernel_launchers_unique_args(
    df_kernel_launchers, include_pct=True
)
df_kernel_launchers_unique_args.head(15)

,name,op category,process_name,process_label,thread_name,Input Dims,Input type,Input Strides,Concrete Inputs,operation_count,total_direct_kernel_time_mean,total_subtree_kernel_time_mean,total_direct_kernel_time_sum,total_subtree_kernel_time_sum,ex_UID,kernel_details_summary,Percentage (%),Cumulative Percentage (%)
0,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 256, 8, 8), (256, 64, 8, 8), (256, 64, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((16384, 64, 8, 1), (4096, 64, 8, 1), (64, 1, ...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",12,564.186910,564.186910,6770.242920,6770.242920,14878,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_MT64x128x32_M...,13.266865,13.266865
1,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 512, 4, 4), (256, 128, 4, 4), (512, 128...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((8192, 16, 4, 1), (2048, 16, 4, 1), (128, 1, ...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",12,306.317871,306.317871,3675.814453,3675.814453,14268,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,7.203070,20.469935
2,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 64, 8, 8), (256, 256, 8, 8), (64, 256, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((4096, 64, 8, 1), (16384, 64, 8, 1), (256, 1,...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",6,584.467244,584.467244,3506.803467,3506.803467,14970,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_MT64x128x32_M...,6.871879,27.341814
3,aten::miopen_convolution,CONV_fwd,python3,CPU,thread 3194582 (python3),"((256, 1024, 2, 2), (256, 1024, 1, 1), (), (),...","(c10::Half, c10::Half, , ScalarList, ScalarLis...","((4096, 4, 2, 1), (1024, 1, 1, 1), (), (), (),...","(, , , [0, 0], [1, 1], [1, 1], 1, True, False)",15,167.320345,167.320345,2509.805176,2509.805176,1738,[{'name': 'miopenSp3AsmConv_v30_3_1_gfx9_fp16_...,4.918176,32.259990
4,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 2048, 1, 1), (256, 1024, 2, 2), (2048, ...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((2048, 1, 1, 1), (4096, 4, 2, 1), (1024, 1, 1...","(, , , [0], [2, 2], [0, 0], [1, 1], False, [0,...",3,677.644694,677.644694,2032.934082,2032.934082,13184,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",3.983707,36.243696
5,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 256, 2, 2), (256, 256, 2, 2), (256, 256...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((1024, 4, 2, 1), (1024, 4, 2, 1), (2304, 9, 3...","(, , , [0], [1, 1], [1, 1], [1, 1], False, [0,...",15,99.387484,99.387484,1490.812256,1490.812256,13417,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",2.921373,39.165069
6,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 64, 8, 8), (256, 64, 8, 8), (64, 64, 1,...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((4096, 64, 8, 1), (4096, 64, 8, 1), (64, 1, 1...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",3,469.175944,469.175944,1407.527832,1407.527832,15296,[{'name': 'Cijk_Ailk_Bjlk_HHS_BH_MT64x32x32_MI...,2.758170,41.923240
7,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 1024, 2, 2), (256, 256, 2, 2), (1024, 2...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((4096, 4, 2, 1), (1024, 4, 2, 1), (256, 1, 1,...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",18,59.312649,59.312649,1067.627686,1067.627686,13371,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",2.092107,44.015347
8,aten::convolution_backward,CONV_bwd,python3,CPU,thread 3196065 (pt_autograd_0),"((256, 256, 2, 2), (256, 1024, 2, 2), (256, 10...","(c10::Half, c10::Half, c10::Half, ScalarList, ...","((1024, 4, 2, 1), (4096, 4, 2, 1), (1024, 1, 1...","(, , , [0], [1, 1], [0, 0], [1, 1], False, [0,...",15,59.966390,59.966390,899.495850,899.495850,13463,"[{'name': 'SubTensorOpWithScalar1d', 'stream':...",

## Replay a Single Event

Pick an event from the table above by its row index. EventReplayer will reconstruct the op's arguments (tensor shapes, dtypes, strides) and execute the operation on the GPU with randomized data.

In [ ]:
# pick an event to replay based on row index from the table above
# change row_idx to replay a different op
device = "cuda"
row_idx = 1 #taking the second most time consuming operation
row = df_kernel_launchers_unique_args.iloc[row_idx]
print(f"Selected op: {row['name']}")

uid = row["ex_UID"]  # get uid for row of interest
evt_to_replay = perf_analyzer.tree.get_UID2event(uid)

# Initialize and replay
my_replayer = EventReplayer(evt_to_replay, device=device, verbose=True)
my_replayer.replay()

Selected op: aten::convolution_backward
Preparing aten::convolution_backward event for replay
Found 2 schemas for aten::convolution_backward:
('aten::convolution_backward(Tensor grad_output, Tensor input, Tensor weight, '
 'SymInt[]? bias_sizes, SymInt[] stride, SymInt[] padding, SymInt[] dilation, '
 'bool transposed, SymInt[] output_padding, SymInt groups, bool[3] '
 'output_mask) -> (Tensor, Tensor, Tensor)')
('aten::convolution_backward.out(Tensor grad_output, Tensor input, Tensor '
 'weight, SymInt[]? bias_sizes, SymInt[] stride, SymInt[] padding, SymInt[] '
 'dilation, bool transposed, SymInt[] output_padding, SymInt groups, bool[3] '
 'output_mask, *, Tensor(a!) out0, Tensor(b!) out1, Tensor(c!) out2) -> '
 '(Tensor(a!), Tensor(b!), Tensor(c!))')
--------------------------------------------------------------------------------
Checking schema:
('aten::convolution_backward(Tensor grad_output, Tensor input, Tensor weight, '
 'SymInt[]? bias_sizes, SymInt[] stride, SymInt[] padding,

### Inspect the Repro Info

`get_repro_info()` returns the lightweight IR — the extracted operator name, argument shapes, dtypes, strides, and concrete values. This is very useful for understanding exactly what args were passed to the op.

In [10]:
# very useful for understanding the op args
my_replayer.get_repro_info()

{'op_name': 'aten::convolution_backward',
 'replay_ir': {'list_pos_args': [{'arg_name': 'grad_output',
    'arg_type': 'Tensor',
    'value': {'shape': [256, 512, 4, 4],
     'dtype': 'c10::Half',
     'strides': [8192, 16, 4, 1],
     'init': 'normal'}},
   {'arg_name': 'input',
    'arg_type': 'Tensor',
    'value': {'shape': [256, 128, 4, 4],
     'dtype': 'c10::Half',
     'strides': [2048, 16, 4, 1],
     'init': 'normal'}},
   {'arg_name': 'weight',
    'arg_type': 'Tensor',
    'value': {'shape': [512, 128, 1, 1],
     'dtype': 'c10::Half',
     'strides': [128, 1, 1, 1],
     'init': 'normal'}},
   {'arg_name': 'bias_sizes', 'arg_type': 'SymInt[]?', 'value': [0]},
   {'arg_name': 'stride', 'arg_type': 'SymInt[]', 'value': [1, 1]},
   {'arg_name': 'padding', 'arg_type': 'SymInt[]', 'value': [0, 0]},
   {'arg_name': 'dilation', 'arg_type': 'SymInt[]', 'value': [1, 1]},
   {'arg_name': 'transposed', 'arg_type': 'bool', 'value': False},
   {'arg_name': 'output_padding', 'arg_type':

### Benchmark the Replayed Op

Quick benchmark to check the fidelity of the replayed op vs the original profile time. 

**Note:** Event Replay uses randomized data based on extracted tensor shapes, so replay timings approximate real-world performance.

In [11]:
def benchmark_func(func, device, warmup=50, avg_steps=100):
    """
    Benchmark a function with warmup and average steps.
    Disclaimer: This method would be inaccurate for very short ops.
    Args:
        func (callable): The function to benchmark.
        warmup (int): Number of warmup iterations.
        avg_steps (int): Number of iterations to average over.
    Returns:
        float: Average time taken per iteration in microseconds.
    """
    # Warmup phase
    for _ in range(warmup):
        func()

    # Benchmarking phase
    torch.cuda.synchronize(device)
    start_time = time.time()
    for _ in range(avg_steps):
        func()
    torch.cuda.synchronize(device)
    end_time = time.time()

    elapsed_time = end_time - start_time
    avg_time_sec = elapsed_time / avg_steps
    avg_time_us = avg_time_sec * 1e6

    return avg_time_us

In [12]:
# Check fidelity of replay vs original profile
replay_time_mean = benchmark_func(my_replayer.replay, device)
profile_time_mean = row["total_direct_kernel_time_mean"]
percent_diff = (replay_time_mean - profile_time_mean) / profile_time_mean * 100
print(f"Average time per replay: {replay_time_mean:.2f} us")
print(f"Profile time mean: {profile_time_mean:.2f} us")
print(f"Percent difference: {percent_diff:.2f}%")
print(f"Abs difference: {replay_time_mean - profile_time_mean:.2f} us")

Average time per replay: 360.75 us
Profile time mean: 306.32 us
Percent difference: 17.77%
Abs difference: 54.43 us


### Verify Replay Fidelity via Profiling

We can further verify that the replay launches the exact same GPU kernels as the original trace by profiling the replay itself and comparing kernel names.

In [13]:
def profile_the_replay(replayer, path="replay_trace.json"):
    """
    Profile the replay of an event to verify kernel fidelity.
    Args:
        replayer (EventReplayer): The EventReplayer object.
        path (str): Output path for the replay trace.
    Returns:
        str: path of the replayed events trace
    """
    def trace_handler(p):
        p.export_chrome_trace(path)

    activities = [ProfilerActivity.CPU, ProfilerActivity.CUDA]
    wait = 10
    warmup = 5
    active = 10
    with profile(
        activities=activities,
        schedule=torch.profiler.schedule(
            wait=wait, warmup=warmup, active=active, repeat=1
        ),
        record_shapes=True,
        on_trace_ready=trace_handler,
    ) as p:
        for idx in range(wait + warmup + active):
            replayer.replay()
            p.step()

    return path

In [14]:
# profile the replay and compare kernel names
replay_profile_path = profile_the_replay(my_replayer)
replay_perf_analyzer = TreePerfAnalyzer.from_file(replay_profile_path)

# find the replayed events in the new trace
replay_evts = [
    e
    for e in replay_perf_analyzer.tree.events
    if e.get("name") == my_replayer.event.get("name")
]
replay_evt = replay_evts[0]

# get kernel names from replay
replay_kernels = [
    replay_perf_analyzer.tree.get_UID2event(uid)
    for uid in replay_evt.get("gpu_events", [])
]
replay_kernels_names = [e.get("name") for e in replay_kernels]

# get kernel names from original (ground truth)
gt_kernels = [
    perf_analyzer.tree.get_UID2event(uid) for uid in evt_to_replay.get("gpu_events", [])
]
gt_kernels_names = [e.get("name") for e in gt_kernels]

print("Ground truth kernels:")
for gt_name in gt_kernels_names:
    print(f"  {gt_name[:128]}")
print()
print("Replay kernels:")
for replay_name in replay_kernels_names:
    print(f"  {replay_name[:128]}")
print()

assert set(replay_kernels_names) == set(
    gt_kernels_names
), f"Replay kernels: {set(replay_kernels_names)} do not match ground truth kernels: {set(gt_kernels_names)}"
print("Kernel names match!")

Building tree with add_python_func=False
Building CPU op tree with add_python_func=False
Ground truth kernels:
  miopenSp3AsmConv_v30_3_1_gfx9_fp16_dot2_edc_f3x2_stride1
  miopenSp3AsmConv_v30_3_1_gfx9_fp16_dot2_edc_f2x3_stride1

Replay kernels:
  miopenSp3AsmConv_v30_3_1_gfx9_fp16_dot2_edc_f3x2_stride1
  miopenSp3AsmConv_v30_3_1_gfx9_fp16_dot2_edc_f2x3_stride1

Kernel names match!


[W417 08:43:32.289163928 collection.cpp:1085] Warning: ROCTracer produced duplicate flow start: 22 (function operator())


## Batched Replay

Instead of replaying one event at a time, we can extract the IR for multiple events and replay them in batch. This is useful for regression testing or comparing performance across hardware.

Let's replay all `aten::convolution_backward` ops from the trace.

In [15]:
# collect all conv backward ops for batched replay
ops_interest = []
df_kernel_launchers_filt = df_kernel_launchers_unique_args[
    df_kernel_launchers_unique_args["name"] == "aten::convolution_backward"
]
for index, row in df_kernel_launchers_filt.iterrows():
    uid = row["ex_UID"]  # get uid for row of interest
    event = perf_analyzer.tree.get_UID2event(uid)
    ops_interest.append(event)

print(f"Found {len(ops_interest)} unique convolution_backward ops to replay")

Found 23 unique convolution_backward ops to replay


In [ ]:
# extract replay IR for each event and save to JSON
# lazy=True means we only extract the IR, no tensor creation yet
repro_data_list = []
processed_count = 0

for event in ops_interest:
    replayer = EventReplayer(
        event, lazy=True, verbose=False
    )  # Set verbose=True for debug

    # Extract the serializable info
    repro_info = replayer.get_repro_info()
    repro_data_list.append(repro_info)
    processed_count += 1

# --- Save the Extracted Data ---
OUTPUT_REPRO_FILE = "event_replay_ir.json"
if repro_data_list:
    print(
        f"\nSaving {len(repro_data_list)} extracted operator infos to '{OUTPUT_REPRO_FILE}'..."
    )
    with open(OUTPUT_REPRO_FILE, "w") as f:
        json.dump(repro_data_list, f, indent=4)
    print("Save complete.")

abs_path_replay_ir_json = os.path.abspath(OUTPUT_REPRO_FILE)
print(f"Repro data saved to: {abs_path_replay_ir_json}")

In [ ]:
# run the batched replay using TraceLens's batched_replay.py script
import subprocess
from TraceLens import EventReplay

dir_batched_replay = os.path.dirname(EventReplay.__file__)
batched_replay_file = os.path.join(dir_batched_replay, "batched_replay.py")
print(f"Running batched replay from directory: {dir_batched_replay}")

cmd = [
    "python",
    batched_replay_file,
    abs_path_replay_ir_json,
    "--verbose",
]
result = subprocess.run(cmd, cwd=dir_batched_replay, capture_output=True, text=True)
if result.returncode != 0:
    print(f"Error running batched replay: {result.stderr}")
else:
    print("Batched replay completed successfully.")
    print(result.stdout)

## Create Standalone Replay Artifacts

Package the replay IR and scripts into a standalone zip file for sharing. These artifacts are **independent of the model code and TraceLens** — anyone with PyTorch can run them.

Artifacts included:
- `event_replay_ir.json` — Serialized operator replay instructions
- `utils.py` — Tensor creation and helper utilities
- `batched_replay.py` — Script to batch replay and benchmark operations
- `batched_replay_readme.md` — Instructions for running the replay

In [ ]:
import zipfile

utils_file_path = os.path.join(dir_batched_replay, "utils.py")
batched_replay_file = os.path.join(dir_batched_replay, "batched_replay.py")
readme_file_path = os.path.join(dir_batched_replay, "batched_replay_readme.md")

files = [
    abs_path_replay_ir_json,  # Path to the replay IR file
    utils_file_path,          # Path to utils.py
    batched_replay_file,      # Path to batched_replay.py
    readme_file_path,         # Path to the readme
]

zip_file_path = "replay_code.zip"
with zipfile.ZipFile(zip_file_path, "w") as zipf:
    for file in files:
        if os.path.exists(file):
            zipf.write(file, arcname=os.path.basename(file))
        else:
            print(f"Warning: {file} not found, skipping")

print(f"Created zip file: {os.path.abspath(zip_file_path)}")